<a href="https://colab.research.google.com/github/zhangling297/deep-learning-with-python-notebooks/blob/master/Cs599_Assignment1_Perceptron_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. A regular perceptron
Created by Ling Zhang
Date: 02-15-2026

In [ ]:
import argparse
import os
import re
from collections import Counter
import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
import sys # Added import sys

SEED = 42
N_SAMPLES = 5000

def clean_text(s: str) -> str:
    s = str(s).lower()
    s = re.sub(r"<br\s*/?>", " ", s)
    s = re.sub(r"[^a-z0-9\s'!?]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def tokenize(text: str):
    return text.split()

def load_5000(path: str):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing dataset: {path}")

    df = pd.read_csv(path)
    if "review" not in df.columns or "sentiment" not in df.columns:
        raise ValueError("CSV must contain columns: review, sentiment")

    df = df.sample(frac=1.0, random_state=SEED).reset_index(drop=True).iloc[:N_SAMPLES]
    texts = df["review"].map(clean_text).to_list()
    y = df["sentiment"].str.lower().map({"negative": -1, "positive": 1}).to_numpy(dtype=int)
    return texts, y

def split_80_10_10(texts, y):
    X_tmp, X_test, y_tmp, y_test = train_test_split(
        texts, y, test_size=0.10, random_state=SEED, stratify=y
    )
    X_train, X_dev, y_train, y_dev = train_test_split(
        X_tmp, y_tmp, test_size=0.1111111111, random_state=SEED, stratify=y_tmp
    )
    return X_train, y_train, X_dev, y_dev, X_test, y_test

def build_vocab(texts, max_vocab=40000, min_df=2):
    """
    Build a vocabulary dict: token -> column index.
    Uses document frequency (df) filtering, then keeps top max_vocab by term frequency.
    """
    df_counter = Counter()
    tf_counter = Counter()

    for t in texts:
        toks = tokenize(t)
        tf_counter.update(toks)
        df_counter.update(set(toks))

    # filter by min_df
    candidates = [w for w, df in df_counter.items() if df >= min_df]
    # rank by term frequency among candidates
    candidates.sort(key=lambda w: tf_counter[w], reverse=True)
    candidates = candidates[:max_vocab]

    vocab = {w: i for i, w in enumerate(candidates)}
    return vocab

def vectorize(text, vocab):
    """
    Return a 1 x |vocab| CSR row vector with raw counts.
    """
    counts = Counter(tokenize(text))
    idx = []
    val = []
    for w, c in counts.items():
        j = vocab.get(w)
        if j is not None:
            idx.append(j)
            val.append(float(c))
    if not idx:
        return sp.csr_matrix((1, len(vocab)), dtype=np.float32)

    idx = np.array(idx, dtype=np.int32)
    val = np.array(val, dtype=np.float32)
    indptr = np.array([0, len(idx)], dtype=np.int32)
    return sp.csr_matrix((val, idx, indptr), shape=(1, len(vocab)), dtype=np.float32)

def add_bias(X):
    """Append a bias feature (constant 1) to sparse matrix."""
    ones = sp.csr_matrix(np.ones((X.shape[0], 1), dtype=np.float32))
    return sp.hstack([X, ones], format="csr")

def decision(xi, w, b):
    score = (xi @ w).item() + b # Use .item() to explicitly get scalar from array
    return 1 if score >= 0 else -1

def perceptron_train(X, y, max_epochs=100, seed=SEED):
    """
    Regular perceptron with explicit bias b (separate from features).
    X is CSR, y in {-1,+1}.
    """
    n, d = X.shape
    w = np.zeros(d, dtype=np.float32)
    b = 0.0

    rng = np.random.default_rng(seed)
    for ep in range(max_epochs):
        idx = np.arange(n)
        rng.shuffle(idx)
        mistakes = 0

        for i in idx:
            xi = X.getrow(i)
            yi = int(y[i])
            # Use .item() to explicitly get scalar from array before comparison
            if yi * ((xi @ w).item() + b) <= 0:
                # w += yi*xi
                w += (yi * xi).toarray().ravel().astype(np.float32)
                b += float(yi)
                mistakes += 1

        if mistakes == 0:
            break

    return w, b

def eval_model(X, y, w, b):
    preds = np.array([decision(X.getrow(i), w, b) for i in range(X.shape[0])], dtype=int)
    acc = accuracy_score(y, preds)
    f1 = f1_score(y, preds, pos_label=1)
    return preds, acc, f1

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default="IMDB Dataset.csv")
    ap.add_argument("--max-vocab", type=int, default=40000)
    ap.add_argument("--min-df", type=int, default=2)
    ap.add_argument("--max-epochs", type=int, default=100)
    ap.add_argument("--print-n", type=int, default=50, help="How many predictions/labels to print")
    args, unknown = ap.parse_known_args() # Modified to parse known arguments and ignore others

    texts, y = load_5000(args.data)
    tr_txt, y_tr, dev_txt, y_dev, te_txt, y_te = split_80_10_10(texts, y)

    # Build vocab on TRAIN ONLY (dev/test never exposed)
    vocab = build_vocab(tr_txt, max_vocab=args.max_vocab, min_df=args.min_df)

    X_tr = sp.vstack([vectorize(t, vocab) for t in tr_txt]).tocsr()
    X_dev = sp.vstack([vectorize(t, vocab) for t in dev_txt]).tocsr()
    X_te = sp.vstack([vectorize(t, vocab) for t in te_txt]).tocsr()

    # add bias feature; we also keep b separately (both work; harmless)

    X_tr = add_bias(X_tr)
    X_dev = add_bias(X_dev)
    X_te = add_bias(X_te)

    print(f"train/dev/test sizes: {X_tr.shape[0]}/{X_dev.shape[0]}/{X_te.shape[0]} | dim={X_tr.shape[1]}")

    # ---- required integrated snippet pattern ----
    w, b = perceptron_train(X_tr, y_tr, max_epochs=args.max_epochs)
    dev_preds, dev_acc, dev_f1 = eval_model(X_dev, y_dev, w, b)
    print(f"DEV accuracy={dev_acc:.4f}  f1={dev_f1:.4f}")

    test_preds, test_acc, test_f1 = eval_model(X_te, y_te, w, b)
    print(f"FINAL TEST accuracy={test_acc:.4f}  f1={test_f1:.4f}")

    # Print like your snippet (but capped by --print-n to avoid huge output)
    n = min(args.print_n, X_te.shape[0])
    preds = test_preds[:n]
    print("Predictions (Yes = +1 = positive), No = -1 = negative):", preds.tolist())
    print("True labels:", y_te[:n].tolist())
    print("w nonzero:", int((w != 0).sum()), " / ", w.size)
    print("b:", b)

if __name__ == "__main__":
    main()

# 2 Voted Perceptron

In [ ]:
#!/usr/bin/env python3
import argparse
import os
import re
from collections import Counter
import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

SEED = 42
N_SAMPLES = 5000

def clean_text(s: str) -> str:
    s = str(s).lower()
    s = re.sub(r"<br\s*/?>", " ", s)
    s = re.sub(r"[^a-z0-9\s'!?]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def tokenize(text: str):
    return text.split()

def load_5000(path: str):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing dataset: {path}")

    df = pd.read_csv(path)
    if "review" not in df.columns or "sentiment" not in df.columns:
        raise ValueError("CSV must contain columns: review, sentiment")

    df = df.sample(frac=1.0, random_state=SEED).reset_index(drop=True).iloc[:N_SAMPLES]
    texts = df["review"].map(clean_text).to_list()
    y = df["sentiment"].str.lower().map({"negative": -1, "positive": 1}).to_numpy(dtype=int)
    return texts, y

def split_80_10_10(texts, y):
    X_tmp, X_test, y_tmp, y_test = train_test_split(
        texts, y, test_size=0.10, random_state=SEED, stratify=y
    )
    X_train, X_dev, y_train, y_dev = train_test_split(
        X_tmp, y_tmp, test_size=0.1111111111, random_state=SEED, stratify=y_tmp
    )
    return X_train, y_train, X_dev, y_dev, X_test, y_test

def build_vocab(texts, max_vocab=40000, min_df=2):
    df_counter = Counter()
    tf_counter = Counter()
    for t in texts:
        toks = tokenize(t)
        tf_counter.update(toks)
        df_counter.update(set(toks))
    candidates = [w for w, df in df_counter.items() if df >= min_df]
    candidates.sort(key=lambda w: tf_counter[w], reverse=True)
    candidates = candidates[:max_vocab]
    return {w: i for i, w in enumerate(candidates)}

def vectorize(text, vocab):
    counts = Counter(tokenize(text))
    idx, val = [], []
    for w, c in counts.items():
        j = vocab.get(w)
        if j is not None:
            idx.append(j)
            val.append(float(c))
    if not idx:
        return sp.csr_matrix((1, len(vocab)), dtype=np.float32)
    idx = np.array(idx, dtype=np.int32)
    val = np.array(val, dtype=np.float32)
    indptr = np.array([0, len(idx)], dtype=np.int32)
    return sp.csr_matrix((val, idx, indptr), shape=(1, len(vocab)), dtype=np.float32)

def add_bias(X):
    ones = sp.csr_matrix(np.ones((X.shape[0], 1), dtype=np.float32))
    return sp.hstack([X, ones], format="csr")

def decision(xi, w, b):
    score = (xi @ w).item() + float(b) # Use .item() here
    return 1 if score >= 0 else -1

def voted_decision(xi, Ws, Bs, Cs):
    vote = 0.0
    for w, b, c in zip(Ws, Bs, Cs):
        vote += float(c) * float(decision(xi, w, b))
    return 1 if vote >= 0 else -1

def voted_predict(X, Ws, Bs, Cs):
    return np.array([voted_decision(X.getrow(i), Ws, Bs, Cs) for i in range(X.shape[0])], dtype=int)

def voted_perceptron_train(X, y, max_epochs=100, seed=SEED):
    """
    Voted perceptron: stores sequence of (w_k, b_k, c_k).
    """
    n, d = X.shape
    w = np.zeros(d, dtype=np.float32)
    b = 0.0
    c = 1

    Ws, Bs, Cs = [], [], []

    rng = np.random.default_rng(seed)
    for ep in range(max_epochs):
        idx = np.arange(n)
        rng.shuffle(idx)
        mistakes = 0

        for i in idx:
            xi = X.getrow(i)
            yi = int(y[i])
            if yi * ((xi @ w).item() + b) <= 0: # Use .item() here
                # save current (w,b) with count c
                Ws.append(w.copy())
                Bs.append(float(b))
                Cs.append(int(c))

                # update
                w += (yi * xi).toarray().ravel().astype(np.float32)
                b += float(yi)

                c = 1
                mistakes += 1
            else:
                c += 1

        if mistakes == 0:
            break

    # include last state too
    Ws.append(w.copy())
    Bs.append(float(b))
    Cs.append(int(c))

    return Ws, Bs, Cs

def eval_model(X, y, preds):
    acc = accuracy_score(y, preds)
    f1 = f1_score(y, preds, pos_label=1)
    return acc, f1

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default="IMDB Dataset.csv")
    ap.add_argument("--max-vocab", type=int, default=40000)
    ap.add_argument("--min-df", type=int, default=2)
    ap.add_argument("--max-epochs", type=int, default=100)
    ap.add_argument("--print-n", type=int, default=50)
    args, unknown = ap.parse_known_args() # Modified to use parse_known_args()

    texts, y = load_5000(args.data)
    tr_txt, y_tr, dev_txt, y_dev, te_txt, y_te = split_80_10_10(texts, y)

    vocab = build_vocab(tr_txt, max_vocab=args.max_vocab, min_df=args.min_df)

    X_tr = sp.vstack([vectorize(t, vocab) for t in tr_txt]).tocsr()
    X_dev = sp.vstack([vectorize(t, vocab) for t in dev_txt]).tocsr()
    X_te = sp.vstack([vectorize(t, vocab) for t in te_txt]).tocsr()

    X_tr = add_bias(X_tr)
    X_dev = add_bias(X_dev)
    X_te = add_bias(X_te)

    print(f"train/dev/test sizes: {X_tr.shape[0]}/{X_dev.shape[0]}/{X_te.shape[0]} | dim={X_tr.shape[1]}")

    # ---- integrated “snippet style” for this model ----
    Ws, Bs, Cs = voted_perceptron_train(X_tr, y_tr, max_epochs=args.max_epochs)

    dev_preds = voted_predict(X_dev, Ws, Bs, Cs)
    dev_acc, dev_f1 = eval_model(X_dev, y_dev, dev_preds)
    print(f"DEV accuracy={dev_acc:.4f}  f1={dev_f1:.4f}  models={len(Ws)}")

    test_preds = voted_predict(X_te, Ws, Bs, Cs)
    test_acc, test_f1 = eval_model(X_te, y_te, test_preds)
    print(f"FINAL TEST accuracy={test_acc:.4f}  f1={test_f1:.4f}")

    # For snippet fields w,b: show the LAST stored (w,b)
    w = Ws[-1]
    b = Bs[-1]

    n = min(args.print_n, X_te.shape[0])
    preds = test_preds[:n]
    print("Predictions (Yes = +1 = positive), No = -1 = negative):", preds.tolist())
    print("True labels:", y_te[:n].tolist())
    print("w nonzero:", int((w != 0).sum()), " / ", w.size)
    print("b:", b)

if __name__ == "__main__":
    main()

# 3. Averaged_perceptron

In [ ]:
import re
import os
import argparse
from collections import Counter
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

In [ ]:
SEED = 42
N_SAMPLES = 5000

def clean_text(s: str) -> str:
    s = str(s).lower()
    s = re.sub(r"<br\s*/?>", " ", s)
    s = re.sub(r"[^a-z0-9\s'!?]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def tokenize(text: str):
    return text.split()

def load_5000(path: str):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing dataset: {path}")

    df = pd.read_csv(path)
    if "review" not in df.columns or "sentiment" not in df.columns:
        raise ValueError("CSV must contain columns: review, sentiment")

    df = df.sample(frac=1.0, random_state=SEED).reset_index(drop=True).iloc[:N_SAMPLES]
    texts = df["review"].map(clean_text).to_list()
    y = df["sentiment"].str.lower().map({"negative": -1, "positive": 1}).to_numpy(dtype=int)
    return texts, y

def split_80_10_10(texts, y):
    X_tmp, X_test, y_tmp, y_test = train_test_split(
        texts, y, test_size=0.10, random_state=SEED, stratify=y
    )
    X_train, X_dev, y_train, y_dev = train_test_split(
        X_tmp, y_tmp, test_size=0.1111111111, random_state=SEED, stratify=y_tmp
    )
    return X_train, y_train, X_dev, y_dev, X_test, y_test

def build_vocab(texts, max_vocab=40000, min_df=2):
    df_counter = Counter()
    tf_counter = Counter()
    for t in texts:
        toks = tokenize(t)
        tf_counter.update(toks)
        df_counter.update(set(toks))
    candidates = [w for w, df in df_counter.items() if df >= min_df]
    candidates.sort(key=lambda w: tf_counter[w], reverse=True)
    candidates = candidates[:max_vocab]
    return {w: i for i, w in enumerate(candidates)}

def vectorize(text, vocab):
    counts = Counter(tokenize(text))
    idx, val = [], []
    for w, c in counts.items():
        j = vocab.get(w)
        if j is not None:
            idx.append(j)
            val.append(float(c))
    if not idx:
        return sp.csr_matrix((1, len(vocab)), dtype=np.float32)
    idx = np.array(idx, dtype=np.int32)
    val = np.array(val, dtype=np.float32)
    indptr = np.array([0, len(idx)], dtype=np.int32)
    return sp.csr_matrix((val, idx, indptr), shape=(1, len(vocab)), dtype=np.float32)

def add_bias(X):
    ones = sp.csr_matrix(np.ones((X.shape[0], 1), dtype=np.float32))
    return sp.hstack([X, ones], format="csr")

def decision(xi, w, b):
    score = (xi @ w).item() + float(b)
    return 1 if score >= 0 else -1

def averaged_perceptron_train(X, y, max_epochs=100, seed=SEED):
    """
    Averaged perceptron: returns averaged (w_avg, b_avg).
    """
    n, d = X.shape
    w = np.zeros(d, dtype=np.float32)
    b = 0.0

    w_sum = np.zeros(d, dtype=np.float64)
    b_sum = 0.0
    t = 0

    rng = np.random.default_rng(seed)
    for ep in range(max_epochs):
        idx = np.arange(n)
        rng.shuffle(idx)
        mistakes = 0

        for i in idx:
            xi = X.getrow(i)
            yi = int(y[i])
            t += 1

            if yi * ((xi @ w).item() + b) <= 0:
                w += (yi * xi).toarray().ravel().astype(np.float32)
                b += float(yi)
                mistakes += 1

            w_sum += w
            b_sum += b

        if mistakes == 0:
            break

    w_avg = (w_sum / max(t, 1)).astype(np.float32)
    b_avg = (b_sum / max(t, 1))
    return w_avg, b_avg

def eval_model(X, y, w, b):
    preds = np.array([decision(X.getrow(i), w, b) for i in range(X.shape[0])], dtype=int)
    acc = accuracy_score(y, preds)
    f1 = f1_score(y, preds, pos_label=1)
    return preds, acc, f1

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default="IMDB Dataset.csv")
    ap.add_argument("--max-vocab", type=int, default=40000)
    ap.add_argument("--min-df", type=int, default=2)
    ap.add_argument("--max-epochs", type=int, default=100)
    ap.add_argument("--print-n", type=int, default=50)
    args, unknown = ap.parse_known_args()

    texts, y = load_5000(args.data)
    tr_txt, y_tr, dev_txt, y_dev, te_txt, y_te = split_80_10_10(texts, y)

    vocab = build_vocab(tr_txt, max_vocab=args.max_vocab, min_df=args.min_df)

    X_tr = sp.vstack([vectorize(t, vocab) for t in tr_txt]).tocsr()
    X_dev = sp.vstack([vectorize(t, vocab) for t in dev_txt]).tocsr()
    X_te = sp.vstack([vectorize(t, vocab) for t in te_txt]).tocsr()

    X_tr = add_bias(X_tr)
    X_dev = add_bias(X_dev)
    X_te = add_bias(X_te)

    print(f"train/dev/test sizes: {X_tr.shape[0]}/{X_dev.shape[0]}/{X_te.shape[0]} | dim={X_tr.shape[1]}")

    # ---- required integrated snippet pattern ----
    w, b = averaged_perceptron_train(X_tr, y_tr, max_epochs=args.max_epochs)

    dev_preds, dev_acc, dev_f1 = eval_model(X_dev, y_dev, w, b)
    print(f"DEV accuracy={dev_acc:.4f}  f1={dev_f1:.4f}")

    test_preds, test_acc, test_f1 = eval_model(X_te, y_te, w, b)
    print(f"FINAL TEST accuracy={test_acc:.4f}  f1={test_f1:.4f}")

    n = min(args.print_n, X_te.shape[0])
    preds = test_preds[:n]
    print("Predictions (Yes = +1 = positive), No = -1 = negative):", preds.tolist())
    print("True labels:", y_te[:n].tolist())
    print("w nonzero:", int((w != 0).sum()), " / ", w.size)
    print("b:", b)

if __name__ == "__main__":
    main()

*README.TXT*

**Data**
IMDB using 5000 samples randomly chosen;

**Packages**
Pip installed numpy pandas scipy scikit-learn

**MODELs**
Sample data tested on regular, voted /average perceptron and data applied 80/20 split rule;

**Script Description**
1. Shuffle dataset and take 5000 reviews randomly
2. Split to 80% train, 10% dev, and 10% test
3. Only used training gtexts to build vocabulary
4. Vectorize text intoa scipy sparse matrix (bag-of-words counts)
5. Train: regular perceptronl voted perceptronl and averaged perceptron
6. Accuracy and F1 were reported on dev and final test
7. Prent prdictions/labels like the provided snippet (default first 50 items)
Additonal:
8: Max-epochs 100, max-vocab 40000,
